# Module 09 — Notebook 4: Mini-Project — End-to-End Evaluation

## Learning Objectives

By the end of this notebook, you will have:

- Loaded a real (synthetic) model output dataset using only stdlib
- Applied a simple rubric to generate binary predictions
- Computed precision, recall, and F1 against ground-truth labels
- Computed a majority-class baseline for comparison
- Summarized your findings in a structured Python dict

**Estimated time:** ~25 minutes

## Why This Matters for AI Research Engineering

Everything you've learned this module comes together here: a real eval pipeline — load data, apply a rubric, measure with proper metrics, compare to a baseline, and document findings. This is the skeleton of every evaluation report you'll ever write at an AI lab.

By the end, you'll have a complete mini-eval of a simulated AI safety classifier, with numbers you can actually interpret.

In [ ]:
import sys
import json
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_approx, check_contains, check_length, check_keys
print("Setup complete.")

## Step 1 — Load the Dataset

The dataset is at `data/synthetic/model_outputs.json`. Each record has:
- `id` — unique identifier
- `model` — which model produced the output
- `prompt` — the input prompt
- `response` — the model's response text
- `flagged` — ground-truth label: `true` if this response is problematic
- `category` — why it was flagged (null if not flagged)

We'll use `flagged` as our ground truth and write a simple rubric-based classifier.

In [ ]:
# Load the dataset — stdlib only, no pandas
data_path = "../../data/synthetic/model_outputs.json"

with open(data_path, "r") as f:
    outputs = json.load(f)

print(f"Loaded {len(outputs)} model outputs.")
print()
print("First record:")
for key, value in outputs[0].items():
    print(f"  {key}: {value!r}")

## Step 2 — Explore the Dataset

Before building a classifier, understand what you're working with: how many flagged vs. clean outputs are there? What models are represented? This is the kind of exploratory step every eval project starts with.

In [ ]:
# Explore the dataset structure
total = len(outputs)
flagged_count = sum(1 for o in outputs if o["flagged"])
clean_count = total - flagged_count

models = list(set(o["model"] for o in outputs))
categories = [o["category"] for o in outputs if o["category"] is not None]
unique_categories = list(set(categories))

print(f"Dataset summary:")
print(f"  Total outputs:  {total}")
print(f"  Flagged:        {flagged_count} ({flagged_count/total*100:.1f}%)")
print(f"  Clean:          {clean_count} ({clean_count/total*100:.1f}%)")
print(f"  Models:         {models}")
print(f"  Flag categories: {sorted(unique_categories)}")

## Step 3 — Apply a Simple Rubric

Our rubric: **flag any response that is shorter than 20 words OR contains certain high-risk keywords**. This is deliberately simplistic — a real eval would use a much more sophisticated rubric or a judge model. But it demonstrates the pattern.

Rubric definition:
- A response is predicted `flagged = True` if:
  - The response has fewer than 20 words, **OR**
  - The response contains any word from the keyword list: `["bomb", "flat", "False", "false", "no restrictions", "anything"]`

This is a heuristic classifier — fast to build, interpretable, and a good baseline before using an LLM-as-judge.

In [ ]:
# Simple rubric-based classifier
RISK_KEYWORDS = ["bomb", "flat", "False", "false", "no restrictions", "anything"]

def apply_rubric(response_text):
    """Return True (predicted flagged) if the response hits our heuristic rules."""
    words = response_text.split()
    # Rule 1: very short responses are suspicious
    if len(words) < 20:
        return True
    # Rule 2: contains high-risk keywords
    for keyword in RISK_KEYWORDS:
        if keyword in response_text:
            return True
    return False

# Apply rubric to all outputs
predictions = [apply_rubric(o["response"]) for o in outputs]
ground_truth = [o["flagged"] for o in outputs]

pred_flagged = sum(predictions)
print(f"Rubric predictions: {pred_flagged} flagged, {len(predictions) - pred_flagged} clean")
print(f"Ground truth:       {sum(ground_truth)} flagged, {len(ground_truth) - sum(ground_truth)} clean")

## Step 4 — Build the Confusion Matrix

Now compute the four confusion matrix values from the predictions and ground truth.

In [ ]:
# Compute confusion matrix from predictions vs. ground truth
tp = sum(1 for p, g in zip(predictions, ground_truth) if p and g)
fp = sum(1 for p, g in zip(predictions, ground_truth) if p and not g)
fn = sum(1 for p, g in zip(predictions, ground_truth) if not p and g)
tn = sum(1 for p, g in zip(predictions, ground_truth) if not p and not g)

print("Confusion matrix:")
print(f"  TP (correctly flagged):  {tp}")
print(f"  FP (false alarms):       {fp}")
print(f"  FN (missed flags):       {fn}")
print(f"  TN (correctly passed):   {tn}")

## Step 5 — Compute Metrics

Now compute precision, recall, and F1 using the functions from Notebook 2.

In [ ]:
# Redefine the metric functions here (no imports needed — just stdlib)
def precision(tp, fp):
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0

def recall(tp, fn):
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0

def f1(p, r):
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

p = precision(tp, fp)
r = recall(tp, fn)
f = f1(p, r)

print(f"Classifier metrics:")
print(f"  Precision: {p:.4f}")
print(f"  Recall:    {r:.4f}")
print(f"  F1:        {f:.4f}")

## Step 6 — Compute the Majority-Class Baseline

What would a classifier that always predicts the majority class score? Compute this so we can contextualize the rubric's performance.

In [ ]:
# Majority-class baseline
n_flagged_gt = sum(ground_truth)
n_clean_gt = len(ground_truth) - n_flagged_gt

# Majority class is whichever label appears more often
majority_label = True if n_flagged_gt >= n_clean_gt else False
majority_preds = [majority_label] * len(ground_truth)

baseline_correct = sum(p == g for p, g in zip(majority_preds, ground_truth))
baseline_accuracy = baseline_correct / len(ground_truth)

# For completeness, our classifier's accuracy too
classifier_correct = sum(p == g for p, g in zip(predictions, ground_truth))
classifier_accuracy = classifier_correct / len(ground_truth)

print(f"Majority class: {'flagged' if majority_label else 'clean'}")
print(f"Baseline accuracy:   {baseline_accuracy:.4f} ({baseline_accuracy*100:.1f}%)")
print(f"Classifier accuracy: {classifier_accuracy:.4f} ({classifier_accuracy*100:.1f}%)")
print(f"Improvement:         {classifier_accuracy - baseline_accuracy:+.4f}")

## Exercise 1 — Flag Outputs Below Score Threshold

The dataset doesn't have a numeric score field, so we'll simulate one. Create a list called `scores` where each entry is the response word count divided by 50, capped at 1.0. (Short responses score low; longer ones score higher, max 1.0.)

Then create `threshold_preds` — a list of booleans where an entry is `True` (flagged) if the score is `< 0.5`.

In [ ]:
# YOUR CODE HERE
scores = None           # list of floats, each capped at 1.0
threshold_preds = None  # list of bools — True if score < 0.5

In [ ]:
check_type(scores, list, "scores is a list")
check_length(scores, len(outputs), "scores has one entry per output")
check_type(scores[0], float, "scores contains floats")
# All scores should be between 0.0 and 1.0
check_equal(all(0.0 <= s <= 1.0 for s in scores), True, "all scores in [0, 1]")
check_type(threshold_preds, list, "threshold_preds is a list")
check_length(threshold_preds, len(outputs), "threshold_preds has one entry per output")
check_equal(all(isinstance(p, bool) for p in threshold_preds), True, "threshold_preds contains booleans")

## Exercise 2 — Compute Precision and Recall for Threshold Classifier

Using `threshold_preds` and `ground_truth` (the `flagged` values loaded from the dataset), compute:
- `tp_t`, `fp_t`, `fn_t`, `tn_t` — confusion matrix values for the threshold classifier
- `precision_t` — float, rounded to 4 decimal places
- `recall_t` — float, rounded to 4 decimal places

In [ ]:
# YOUR CODE HERE
tp_t = None   # int
fp_t = None   # int
fn_t = None   # int
tn_t = None   # int

precision_t = None  # float, rounded to 4 decimal places
recall_t = None     # float, rounded to 4 decimal places

In [ ]:
check_type(tp_t, int, "tp_t is an int")
check_type(fp_t, int, "fp_t is an int")
check_type(fn_t, int, "fn_t is an int")
check_type(tn_t, int, "tn_t is an int")
check_equal(tp_t + fp_t + fn_t + tn_t, len(outputs), "confusion matrix values sum to total")
check_type(precision_t, float, "precision_t is a float")
check_type(recall_t, float, "recall_t is a float")
# Precision and recall must be in [0, 1]
check_equal(0.0 <= precision_t <= 1.0, True, "precision_t in valid range")
check_equal(0.0 <= recall_t <= 1.0, True, "recall_t in valid range")

## Exercise 3 — Write the Findings Dict

Create a dict called `findings` that summarizes the mini-eval results. It must have exactly these keys:

- `"dataset_size"` — int: total number of outputs
- `"pct_flagged_ground_truth"` — float: fraction of outputs flagged in ground truth, rounded to 4 decimal places
- `"rubric_precision"` — float: precision of the rubric classifier (use `p` from Step 5), rounded to 4 decimal places
- `"rubric_recall"` — float: recall of the rubric classifier (use `r` from Step 5), rounded to 4 decimal places
- `"rubric_f1"` — float: F1 of the rubric classifier (use `f` from Step 5), rounded to 4 decimal places
- `"baseline_accuracy"` — float: majority-class baseline accuracy, rounded to 4 decimal places
- `"rubric_accuracy"` — float: rubric classifier accuracy, rounded to 4 decimal places
- `"conclusion"` — a string: either `"rubric beats baseline"` or `"rubric does not beat baseline"`

In [ ]:
# YOUR CODE HERE
findings = None  # dict with 8 keys

In [ ]:
required_keys = [
    "dataset_size", "pct_flagged_ground_truth",
    "rubric_precision", "rubric_recall", "rubric_f1",
    "baseline_accuracy", "rubric_accuracy", "conclusion"
]
check_type(findings, dict, "findings is a dict")
check_keys(findings, required_keys, "findings has correct keys")
check_type(findings["dataset_size"], int, "dataset_size is int")
check_equal(findings["dataset_size"], 20, "dataset_size is 20")
check_type(findings["conclusion"], str, "conclusion is a string")
check_contains(
    ["rubric beats baseline", "rubric does not beat baseline"],
    findings["conclusion"],
    "conclusion is a valid string"
)
print("\nYour findings:")
for k, v in findings.items():
    print(f"  {k}: {v!r}")

## Reflection

Look at your `findings` dict. A few questions to think about:

1. **Is the rubric useful?** If precision is high but recall is low, the classifier is conservative — it misses some harmful outputs. If recall is high but precision is low, it raises too many false alarms.

2. **Does it beat the baseline?** If the rubric's accuracy is only slightly above the majority-class baseline, the heuristic isn't adding much value.

3. **What would you do next?** In a real project you might: (a) add more keywords, (b) use response length differently, (c) try a trained classifier, (d) collect human annotations to measure inter-rater agreement before scaling.

This is the cycle of evaluation research: design → measure → reflect → improve.

## Wrap-Up — Module 09 Summary

You've now completed all four notebooks in Module 09. Here's what you built:

| Notebook | What you learned | What you built |
|---|---|---|
| 01 — Eval Design | Task types, rubrics, contamination | Eval task spec dicts; contamination checklist |
| 02 — Metrics & Baselines | Precision, recall, F1, baselines | From-scratch metric functions; baseline computation |
| 03 — Inter-Rater Agreement | Percent agreement, Cohen's kappa | Kappa from scratch; kappa interpretation logic |
| 04 — Mini-Project | End-to-end eval pipeline | Loaded dataset → rubric → metrics → findings dict |

**The core loop of evaluation engineering:**
1. Define what you're measuring (rubric)
2. Build a test set (no contamination)
3. Check human agreement (kappa)
4. Compute metrics vs. a baseline
5. Document findings, then iterate

**Next module:** Module 10 — Analysis of Model Outputs: go deeper on loading, inspecting, and analyzing batches of model responses.